In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.nn.parallel import DataParallel
from tqdm import tqdm
import re
from collections import Counter
import pickle
import math

# Hyperparameters
batch_size = 64
block_size = 128  # Reduced since we're using tokens instead of chars
max_iters = 10000
eval_interval = 1000  # Evaluate more frequently
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 512  # Increased embedding dimension
n_head = 8
n_layer = 8
dropout = 0.1
min_token_freq = 2  # Minimum frequency for a token to be included in vocab
warmup_iters = 100  # Warmup steps for learning rate
# ------------

torch.manual_seed(1337)

print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"Number of GPUs available: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

# ============================================================
# TOKENIZER CLASS
# ============================================================
class KabyleTokenizer:
    """Simple word-based tokenizer for Kabyle language"""
    
    def __init__(self, min_freq=2):
        self.min_freq = min_freq
        self.word2idx = {}
        self.idx2word = {}
        self.vocab_size = 0
        
    def build_vocab(self, text):
        """Build vocabulary from text"""
        # Tokenize: split on whitespace and punctuation but keep punctuation
        tokens = re.findall(r'\w+|[^\w\s]', text.lower())
        
        # Count token frequencies
        token_counts = Counter(tokens)
        
        # Add special tokens
        self.word2idx = {
            '<PAD>': 0,
            '<UNK>': 1,
            '<BOS>': 2,  # Beginning of sequence
            '<EOS>': 3,  # End of sequence
        }
        
        # Add tokens that meet minimum frequency
        idx = len(self.word2idx)
        for token, count in token_counts.most_common():
            if count >= self.min_freq:
                self.word2idx[token] = idx
                idx += 1
        
        # Create reverse mapping
        self.idx2word = {idx: word for word, idx in self.word2idx.items()}
        self.vocab_size = len(self.word2idx)
        
        print(f"Vocabulary size: {self.vocab_size}")
        print(f"Total unique tokens: {len(token_counts)}")
        print(f"Tokens with freq >= {self.min_freq}: {self.vocab_size - 4}")
        
    def encode(self, text):
        """Convert text to token indices"""
        tokens = re.findall(r'\w+|[^\w\s]', text.lower())
        return [self.word2idx.get(token, self.word2idx['<UNK>']) for token in tokens]
    
    def decode(self, indices):
        """Convert token indices back to text"""
        tokens = [self.idx2word.get(idx, '<UNK>') for idx in indices]
        # Simple reconstruction: add spaces between words but not before punctuation
        text = []
        for i, token in enumerate(tokens):
            if token in ['<PAD>', '<BOS>', '<EOS>']:
                continue
            if i > 0 and token not in '.,!?;:)\']}"' and tokens[i-1] not in '([{"':
                text.append(' ')
            text.append(token)
        return ''.join(text)
    
    def save(self, path):
        """Save tokenizer to file"""
        with open(path, 'wb') as f:
            pickle.dump({
                'word2idx': self.word2idx,
                'idx2word': self.idx2word,
                'vocab_size': self.vocab_size,
                'min_freq': self.min_freq
            }, f)
    
    def load(self, path):
        """Load tokenizer from file"""
        with open(path, 'rb') as f:
            data = pickle.load(f)
            self.word2idx = data['word2idx']
            self.idx2word = data['idx2word']
            self.vocab_size = data['vocab_size']
            self.min_freq = data['min_freq']

# ============================================================
# DATA LOADING
# ============================================================
print("Loading and tokenizing data...")
with open('/kaggle/input/kab-latn/kabyle_wiki_corpus.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# Build tokenizer
tokenizer = KabyleTokenizer(min_freq=min_token_freq)
tokenizer.build_vocab(text)
tokenizer.save('kabyle_tokenizer.pkl')

# Encode the entire dataset
data = torch.tensor(tokenizer.encode(text), dtype=torch.long)
print(f"Total tokens in dataset: {len(data)}")

# Train and validation splits
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

# ============================================================
# BATCH GENERATION
# ============================================================
def get_batch(split):
    """Generate a batch of data"""
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    """Estimate loss on train and validation sets"""
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            # Handle DataParallel which returns a tensor instead of scalar
            if isinstance(loss, torch.Tensor) and loss.dim() > 0:
                loss = loss.mean()
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

# ============================================================
# MODEL ARCHITECTURE
# ============================================================
class SinusoidalPositionalEmbedding(nn.Module):
    """Sinusoidal positional embeddings (as in 'Attention is All You Need')"""
    
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)
        
    def forward(self, x):
        """x: (batch_size, seq_len, d_model)"""
        return self.pe[:x.size(1), :]

class Head(nn.Module):
    """Single head of self-attention"""
    
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)
        self.head_size = head_size
        
    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        
        # Scaled dot-product attention
        wei = q @ k.transpose(-2, -1) * (self.head_size ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    """Multiple heads of self-attention in parallel"""
    
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedForward(nn.Module):
    """Feed-forward network with GELU activation"""
    
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),  # GELU is better than ReLU for transformers
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )
        
    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    """Transformer block with pre-layer normalization"""
    
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
        
    def forward(self, x):
        # Pre-LN architecture (more stable than post-LN)
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class KabyleGPT(nn.Module):
    """GPT-style language model for Kabyle"""
    
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = SinusoidalPositionalEmbedding(n_embd, max_len=block_size)
        self.blocks = nn.Sequential(*[TransformerBlock(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        
        # Weight tying: share weights between token embedding and output layer
        self.lm_head.weight = self.token_embedding.weight
        
        # Initialize weights
        self.apply(self._init_weights)
        
    def _init_weights(self, module):
        """Robust weight initialization (GPT-2 style)"""
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.LayerNorm):
            torch.nn.init.zeros_(module.bias)
            torch.nn.init.ones_(module.weight)
            
    def forward(self, idx, targets=None):
        B, T = idx.shape
        
        # Token + positional embeddings
        tok_emb = self.token_embedding(idx)
        pos_emb = self.position_embedding(tok_emb)
        x = tok_emb + pos_emb
        
        # Transformer blocks
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
            
        return logits, loss
    
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """Generate text with temperature and top-k sampling"""
        for _ in range(max_new_tokens):
            # Crop context if needed
            idx_cond = idx if idx.size(1) <= block_size else idx[:, -block_size:]
            
            # Get predictions
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            
            # Top-k sampling
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
            
        return idx

# ============================================================
# TRAINING
# ============================================================
print("\nInitializing model...")
model = KabyleGPT(tokenizer.vocab_size)

# Use DataParallel for multi-GPU training
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs with DataParallel")
    model = DataParallel(model)

model = model.to(device)

# Print model size
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params/1e6:.2f}M")

# Optimizer with weight decay
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)

# Learning rate scheduler with warmup
def get_lr(iter):
    """Learning rate schedule with warmup and cosine decay"""
    # Linear warmup
    if iter < warmup_iters:
        return learning_rate * (iter + 1) / warmup_iters
    # Cosine decay
    decay_ratio = (iter - warmup_iters) / (max_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return learning_rate * 0.1 + coeff * (learning_rate - learning_rate * 0.1)

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda iter: get_lr(iter) / learning_rate)

# Sanity check - get initial loss
print("\nRunning initial evaluation...")
initial_losses = estimate_loss()
print(f"Initial train loss: {initial_losses['train']:.4f}")
print(f"Initial val loss: {initial_losses['val']:.4f}")
print(f"Expected loss (random): ~{math.log(tokenizer.vocab_size):.4f}")
print(f"(Loss should start near expected and decrease during training)")

print("\nStarting training...")
pbar = tqdm(range(max_iters), desc="Training")
best_val_loss = float('inf')
train_losses = []  # Track training loss

for iter in pbar:
    # Evaluation
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        
        # Calculate average training loss since last eval
        avg_train_loss = sum(train_losses[-eval_interval:]) / len(train_losses[-eval_interval:]) if train_losses else 0
        
        pbar.set_postfix({
            'train': f"{losses['train']:.4f}",
            'val': f"{losses['val']:.4f}",
            'lr': f"{optimizer.param_groups[0]['lr']:.2e}",
            'step_loss': f"{avg_train_loss:.4f}"
        })
        
        # Print detailed update
        #if iter % (eval_interval * 2) == 0:
         #   print(f"\nIter {iter}: train={losses['train']:.4f}, val={losses['val']:.4f}, lr={optimizer.param_groups[0]['lr']:.2e}")
        
        # Save best model
        if losses['val'] < best_val_loss:
            best_val_loss = losses['val']
            checkpoint = {
                'iter': iter,
                'model_state_dict': model.module.state_dict() if isinstance(model, DataParallel) else model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_loss': losses['train'].item(),
                'val_loss': losses['val'].item(),
                'vocab_size': tokenizer.vocab_size,
            }
            torch.save(checkpoint, 'best_kabyle_model.pth')
    
    # Training step
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    
    # Handle DataParallel which returns a tensor instead of scalar
    if isinstance(loss, torch.Tensor) and loss.dim() > 0:
        loss = loss.mean()
    
    train_losses.append(loss.item())
    
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    
    # Gradient clipping
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    
    optimizer.step()
    scheduler.step()

print("\n" + "="*60)
print("Training completed!")
print(f"Best validation loss: {best_val_loss:.4f}")
print("="*60)

Using device: cuda
Number of GPUs available: 2
GPU 0: Tesla T4
GPU 1: Tesla T4
Loading and tokenizing data...
Vocabulary size: 35207
Total unique tokens: 87720
Tokens with freq >= 2: 35203
Total tokens in dataset: 995159

Initializing model...
Using 2 GPUs with DataParallel
Total parameters: 43.27M

Running initial evaluation...


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Initial train loss: 10.6479
Initial val loss: 10.6493
Expected loss (random): ~10.4690
(Loss should start near expected and decrease during training)

Starting training...


Training:  82%|████████▏ | 8207/10000 [1:25:39<18:42,  1.60it/s, train=1.3286, val=6.2081, lr=5.63e-05, step_loss=1.8073]   


KeyboardInterrupt: 

In [2]:
# ============================================================
# TEXT GENERATION
# ============================================================
print("\nGenerating text...")
# Load best model
checkpoint = torch.load('/kaggle/working/best_kabyle_model.pth')
generation_model = KabyleGPT(tokenizer.vocab_size).to(device)
generation_model.load_state_dict(checkpoint['model_state_dict'])
generation_model.eval()

def generate_text(prompt, max_tokens=100, temperature=0.8, top_k=40):
    """Generate text from a prompt"""
    context = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long, device=device)
    generated = generation_model.generate(context, max_new_tokens=max_tokens, 
                                         temperature=temperature, top_k=top_k)
    return tokenizer.decode(generated[0].tolist()).replace("<UNK>"," ")

# Example generations
prompts = [
    "Azul",
    "Tamurt",
    "Aqcic",
]

print("\n" + "="*60)
print("GENERATED TEXTS")
print("="*60)

for prompt in prompts:
    print(f"\nPrompt: '{prompt}'")
    print("-" * 60)
    generated = generate_text(prompt, max_tokens=80, temperature=0.8, top_k=40)
    print(generated)
    print()

print("\n" + "="*60)
print("Custom prompt generation (enter your own)")
print("="*60)
custom_prompt = input("\nEnter a Kabyle prompt: ")
if custom_prompt:
    print("\nGenerated:")
    print(generate_text(custom_prompt, max_tokens=100, temperature=0.8, top_k=40))


Generating text...

GENERATED TEXTS

Prompt: 'Azul'
------------------------------------------------------------
azul d  . adlis - a d tin i d - yeglan. deg tallit - a, yefka - d yiwet n   n umdan akken ad ḥerzen yiwen seg   - is ad     - d,   n  , d wayen nniḍen s deffir - s,  .     - ed   ɣer yiwen n   n   deg   n  .   akked" "( ) n"


Prompt: 'Tamurt'
------------------------------------------------------------
tamurt n turuft, tezga - d deg wenẓul - agmuḍan (nord - est) n tmurt, ɣef yiri n ubagu n honcu. zedɣen - tt 53, 636 n yimezdaɣen. tamaneɣt - nnes d tamdint n nashville.   d taɣiwant n ugafa - agmuḍ n fransa. zedɣen - tt 103. 364 n yimezdaɣen.   d taɣiwant n twilayt n skikda, zedɣen - tt 116. 724 n yimezdaɣen. deg tallit


Prompt: 'Aqcic'
------------------------------------------------------------
aqcic - is, d ayen i t - yeǧǧan ad   iman - is ɣer  , dɣa d aya ay d tameṭṭut ay d - yeqqimen d   s umata, xas akken yemmut imi ay d - tefka   s wudem - is i wakken ad aɣ - d - awi


Enter a Kabyle prompt:  saɛid saɛdi



Generated:
saɛid saɛdi, acennay  ,  ,  ,  ,  ,  ,  ,  ,  ,  ,  ,  ,  ,  ,  ,  ,  ,  ,  ,  ,  ,  ,  ,  ,  ,  ,  ,  ,  ,  ,  ,  ,  ,  ,  ,  ,  ,  ,  ,  ,  ,  ,  ,  ,  ,  ,  ,  ,  ,


In [ ]:
custom_prompt = input("\nEnter a Kabyle prompt: ")
if custom_prompt:
    print("\nGenerated:")
    print(generate_text(custom_prompt, max_tokens=100, temperature=0.8, top_k=40))


Enter a Kabyle prompt:  i tura akka



Generated:
